In [ ]:
# Mount Google Drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# Hardware Results Plots for Thesis
# 6. Hardware visibility verdict by model
# 7. Hardware ΔE2E latency heatmap
# 8. Hardware ΔPower heatmap
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.patches import Rectangle, Patch
import seaborn as sns

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
# For local Colab session:
OUT_DIR = "/content/hardware_plots"

# For Google Drive, uncomment these two lines:
# from google.colab import drive
# drive.mount('/content/drive')
# OUT_DIR = "/content/drive/MyDrive/thesis_plots/hardware"

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Global plot style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family":       "DejaVu Sans",
    "font.size":         10,
    "axes.titlesize":    12,
    "axes.titleweight":  "bold",
    "axes.labelsize":    10,
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "legend.fontsize":   9,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "pdf.fonttype":      42,   # editable text in Illustrator / Inkscape
    "ps.fonttype":       42,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.grid":         False,
})

# Shared colour palette
C_INVISIBLE = "#1B6CA8"   # blue  – hardware-invisible
C_VISIBLE   = "#C0392B"   # red   – visible footprint
C_BADGE_BG  = "#EEF4FB"   # light blue – summary badge background

# Shared diverging colormap (also looks fine in greyscale)
CMAP = sns.diverging_palette(220, 20, s=65, l=50, as_cmap=True)

models    = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet+", "RT-DETR-L"]
scenarios = [f"B{i:02d}" for i in range(1, 10)]


# ============================================================
# 6. Hardware visibility verdict by model
# ============================================================

invisible = np.array([9, 9, 9, 9, 8])
visible   = np.array([0, 0, 0, 0, 1])

fig, ax = plt.subplots(figsize=(6.8, 3.9))
fig.patch.set_facecolor("white")

x = np.arange(len(models))
w = 0.52

ax.bar(x, invisible, width=w, color=C_INVISIBLE, zorder=3, linewidth=0)
ax.bar(x, visible, bottom=invisible, width=w, color=C_VISIBLE, zorder=3, linewidth=0)

# In-bar labels
for i, (inv, vis) in enumerate(zip(invisible, visible)):
    ax.text(i, inv / 2, f"{inv}/9",
            ha="center", va="center",
            color="white", fontweight="bold", fontsize=11)
    if vis > 0:
        ax.text(i, inv + vis / 2, f"{vis}/9",
                ha="center", va="center",
                color="white", fontweight="bold", fontsize=10)

# Arrow callout for the lone outlier
ax.annotate(
    "DVFS scheduling\nartifact (not patch-\ninduced)",
    xy=(4, 9.07), xytext=(3.08, 9.75),
    arrowprops=dict(arrowstyle="->", color=C_VISIBLE, lw=1.3,
                    connectionstyle="arc3,rad=0.15"),
    fontsize=8, color=C_VISIBLE, ha="center", va="bottom",
    bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
              edgecolor=C_VISIBLE, linewidth=0.8, alpha=0.92),
)

# Axes
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel("Scenarios (out of 9)")
ax.set_ylim(0, 11.0)
ax.spines["left"].set_color("#BBBBBB")
ax.spines["bottom"].set_color("#BBBBBB")
ax.axhline(9, color="#CCCCCC", lw=0.8, ls="--", zorder=1)
ax.set_title("Hardware Visibility Verdict by Model", pad=10)

# Legend
ax.legend(
    handles=[Patch(facecolor=C_INVISIBLE, label="Hardware-invisible"),
             Patch(facecolor=C_VISIBLE,   label="Visible footprint")],
    frameon=False, loc="upper left",
)

# Headline badge
ax.text(
    0.99, 0.97, "44 / 45 hardware-invisible  (98%)",
    transform=ax.transAxes, ha="right", va="top",
    fontsize=9, color=C_INVISIBLE, fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.35", facecolor=C_BADGE_BG,
              edgecolor=C_INVISIBLE, linewidth=0.8),
)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "hardware_visibility_verdict_by_model.pdf"),
            bbox_inches="tight")
plt.savefig(os.path.join(OUT_DIR, "hardware_visibility_verdict_by_model.png"),
            bbox_inches="tight")
plt.show()


# ============================================================
# 7. Hardware ΔE2E latency heatmap
# ΔE2E = patched E2E − clean E2E  (ms)
# ============================================================

delta_e2e = np.array([
    [-3.03, -0.31, +0.49, +2.58, +0.77, +0.50, -0.15, +0.41, +0.65],  # YOLOv8n
    [-0.98, +0.29, -0.14, +0.50, -0.21, -0.05, +0.09, +0.08, +0.06],  # YOLOv8s
    [+1.46, -0.72, -0.86, +1.21, +0.37, -0.78, +0.08, +1.41, -0.77],  # YOLOv5n
    [-1.15, -0.63, +1.10, +0.11, -0.07, -1.00, -1.45, +1.86, +2.94],  # NanoDet+
    [-1.38, -1.16, -3.30, +2.10, +3.23, -2.89, +1.74, -5.03, -0.92],  # RT-DETR-L
])

fig, ax = plt.subplots(figsize=(9.4, 4.0))
fig.patch.set_facecolor("white")

vmax = 5.5
im = ax.imshow(delta_e2e, cmap=CMAP, vmin=-vmax, vmax=vmax, aspect="auto")

ax.set_xticks(np.arange(len(scenarios)))
ax.set_xticklabels(scenarios)
ax.set_yticks(np.arange(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel("Billboard scenario")
ax.set_title("Patched − Clean End-to-End Latency  (ms)", pad=10)

for spine in ax.spines.values():
    spine.set_visible(False)

# Cell annotations
for i in range(delta_e2e.shape[0]):
    for j in range(delta_e2e.shape[1]):
        val = delta_e2e[i, j]
        tc = "white" if abs(val) > 3.2 else "#1A1A1A"
        ax.text(j, i, f"{val:+.2f}", ha="center", va="center",
                color=tc, fontsize=8.5)

# Cell grid lines
ax.set_xticks(np.arange(-.5, len(scenarios), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(models),    1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.6)
ax.tick_params(which="minor", bottom=False, left=False)

cbar = plt.colorbar(im, ax=ax, fraction=0.028, pad=0.02)
cbar.set_label("Δ ms", labelpad=6)
cbar.outline.set_visible(False)

# Caption below figure
fig.text(
    0.5, -0.04,
    "All 45 cells fall within natural run-to-run timing noise — "
    "no consistent patch-induced latency footprint.",
    ha="center", fontsize=8.5, color="#555555", style="italic",
)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "hardware_delta_e2e_heatmap.pdf"),
            bbox_inches="tight")
plt.savefig(os.path.join(OUT_DIR, "hardware_delta_e2e_heatmap.png"),
            bbox_inches="tight")
plt.show()


# ============================================================
# 8. Hardware ΔPower heatmap
# ΔPower = patched power − clean power  (W)
# ============================================================

delta_power = np.array([
    [-0.033, -0.013, +0.024, +0.059, -0.066, -0.002, +0.032, +0.113, +0.000],  # YOLOv8n
    [-0.083, -0.061, +0.017, -0.091, +0.012, -0.004, +0.042, +0.009, +0.002],  # YOLOv8s
    [-0.000, -0.004, -0.000, -0.001, -0.001, +0.000, -0.005, +0.000, +0.026],  # YOLOv5n
    [-0.045, +0.000, -0.005, -0.013, -0.023, +0.010, +0.001, +0.049, +0.028],  # NanoDet+
    [-0.062, -0.088, +0.416, -0.011, +0.094, +0.085, -0.076, -0.047, -0.024],  # RT-DETR-L
])

NOISE_FLOOR = 0.22   # INA3221 sensor noise threshold (W)

fig, ax = plt.subplots(figsize=(9.4, 4.0))
fig.patch.set_facecolor("white")

vmax_p = 0.45
im = ax.imshow(delta_power, cmap=CMAP, vmin=-vmax_p, vmax=vmax_p, aspect="auto")

ax.set_xticks(np.arange(len(scenarios)))
ax.set_xticklabels(scenarios)
ax.set_yticks(np.arange(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel("Billboard scenario")
ax.set_title("Patched − Clean Power Draw  (W)", pad=10)

for spine in ax.spines.values():
    spine.set_visible(False)

# Cell annotations
for i in range(delta_power.shape[0]):
    for j in range(delta_power.shape[1]):
        val = delta_power[i, j]
        tc = "white" if abs(val) > 0.28 else "#1A1A1A"
        ax.text(j, i, f"{val:+.3f}", ha="center", va="center",
                color=tc, fontsize=8)

# Bold outline + dagger for cells that exceed the noise floor
for i in range(delta_power.shape[0]):
    for j in range(delta_power.shape[1]):
        if abs(delta_power[i, j]) > NOISE_FLOOR:
            rect = Rectangle((j - 0.5, i - 0.5), 1, 1,
                              fill=False, edgecolor="#111111",
                              linewidth=2.4, zorder=5)
            ax.add_patch(rect)
            ax.text(j + 0.46, i - 0.40, "†",
                    ha="right", va="top",
                    fontsize=11, color="#111111", fontweight="bold")

# Cell grid lines
ax.set_xticks(np.arange(-.5, len(scenarios), 1), minor=True)
ax.set_yticks(np.arange(-.5, len(models),    1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.6)
ax.tick_params(which="minor", bottom=False, left=False)

cbar = plt.colorbar(im, ax=ax, fraction=0.028, pad=0.02)
cbar.set_label("Δ W", labelpad=6)
cbar.outline.set_visible(False)

# Mark ±noise-floor on the colourbar
for sign in (+1, -1):
    cbar.ax.axhline(
        0.5 + sign * (NOISE_FLOOR / vmax_p) * 0.5,
        color="#333333", lw=1.0, ls="--",
    )

fig.text(
    0.5, -0.04,
    "† RT-DETR-L / B03 is the only cell exceeding the INA3221 noise floor (±0.22 W). "
    "This is a DVFS scheduling artifact — not a patch-induced power change.",
    ha="center", fontsize=8.5, color="#555555", style="italic",
)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "hardware_delta_power_heatmap.pdf"),
            bbox_inches="tight")
plt.savefig(os.path.join(OUT_DIR, "hardware_delta_power_heatmap.png"),
            bbox_inches="tight")
plt.show()


print(f"\nAll plots saved to: {OUT_DIR}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
OUT_DIR = "/content/hardware_plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Data: YOLOv8n pipeline breakdown
# ------------------------------------------------------------
labels = ["Clean", "Patched"]
preprocessing = np.array([26.4, 28.3])
engine = np.array([37.0, 37.0])
nms = np.array([5.0, 5.3])

totals = preprocessing + engine + nms
delta_total = totals[1] - totals[0]

# ------------------------------------------------------------
# Style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 16,
    "legend.fontsize": 14,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

C_PRE = "#C7D3DF"
C_ENG = "#6F96BD"
C_NMS = "#2B5A88"

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.2, 3.2))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

ypos = np.array([1, 0])   # Clean on top, Patched below
bar_h = 0.38

# ------------------------------------------------------------
# Stacked horizontal bars
# ------------------------------------------------------------
ax.barh(
    ypos, preprocessing,
    height=bar_h,
    color=C_PRE,
    edgecolor="white",
    linewidth=1.0,
    zorder=3
)

ax.barh(
    ypos, engine,
    left=preprocessing,
    height=bar_h,
    color=C_ENG,
    edgecolor="white",
    linewidth=1.0,
    zorder=3
)

ax.barh(
    ypos, nms,
    left=preprocessing + engine,
    height=bar_h,
    color=C_NMS,
    edgecolor="white",
    linewidth=1.0,
    zorder=3
)

# ------------------------------------------------------------
# Segment labels
# ------------------------------------------------------------
for i in range(2):
    # preprocessing
    ax.text(
        preprocessing[i] / 2,
        ypos[i],
        f"{preprocessing[i]:.1f}",
        ha="center",
        va="center",
        color="#222222",
        fontsize=13
    )

    # engine
    ax.text(
        preprocessing[i] + engine[i] / 2,
        ypos[i],
        f"{engine[i]:.1f}",
        ha="center",
        va="center",
        color="white",
        fontsize=13,
        fontweight="bold"
    )

    # NMS
    ax.text(
        preprocessing[i] + engine[i] + nms[i] / 2,
        ypos[i],
        f"{nms[i]:.1f}",
        ha="center",
        va="center",
        color="white",
        fontsize=13,
        fontweight="bold"
    )

    # total label
    ax.text(
        totals[i] + 1.0,
        ypos[i],
        f"{totals[i]:.1f} ms",
        ha="left",
        va="center",
        fontsize=16,
        fontweight="bold",
        color="#111111"
    )

# ------------------------------------------------------------
# Bracket showing total delta
# ------------------------------------------------------------
x_bracket = max(totals) + 2.7
tick_len = 0.5

y_top = ypos[0] - bar_h / 2
y_bottom = ypos[1] + bar_h / 2

ax.plot([x_bracket, x_bracket], [y_bottom, y_top], color="#777777", lw=1.5)
ax.plot([x_bracket - tick_len, x_bracket], [y_top, y_top], color="#777777", lw=1.5)
ax.plot([x_bracket - tick_len, x_bracket], [y_bottom, y_bottom], color="#777777", lw=1.5)

ax.text(
    x_bracket + 0.6,
    (y_top + y_bottom) / 2,
    f"+{delta_total:.1f} ms",
    ha="left",
    va="center",
    fontsize=16,
    fontweight="bold",
    color="#333333"
)

# ------------------------------------------------------------
# Axes formatting
# ------------------------------------------------------------
ax.set_yticks(ypos)
ax.set_yticklabels(labels)

ax.set_xlabel("Latency (ms)")
ax.set_xlim(0, 80)
ax.set_xticks(np.arange(0, 81, 10))

ax.grid(axis="x", linestyle="--", linewidth=0.8, alpha=0.35, zorder=0)
ax.grid(axis="y", visible=False)

ax.spines["left"].set_color("#333333")
ax.spines["bottom"].set_color("#333333")
ax.spines["left"].set_linewidth(1.1)
ax.spines["bottom"].set_linewidth(1.1)

# ------------------------------------------------------------
# Legend above
# ------------------------------------------------------------
legend_handles = [
    Patch(facecolor=C_PRE, edgecolor="none", label="Preprocessing"),
    Patch(facecolor=C_ENG, edgecolor="none", label="Engine"),
    Patch(facecolor=C_NMS, edgecolor="none", label="NMS"),
]

ax.legend(
    handles=legend_handles,
    ncol=3,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, 1.20)
)

# ------------------------------------------------------------
# Save ONCE before plt.show()
# ------------------------------------------------------------
png_path = os.path.join(OUT_DIR, "yolov8n_pipeline_breakdown_final.png")
pdf_path = os.path.join(OUT_DIR, "yolov8n_pipeline_breakdown_final.pdf")

fig.tight_layout()
fig.subplots_adjust(top=0.80)

fig.savefig(png_path, format="png", bbox_inches="tight", dpi=300, facecolor="white")
fig.savefig(pdf_path, format="pdf", bbox_inches="tight", facecolor="white")

plt.show()

print("Saved PNG:", png_path)
print("Saved PDF:", pdf_path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
OUT_DIR = "/content/hardware_plots_cleaner"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet+", "RT-DETR-L"]

clean_mean   = np.array([45.767, 60.846, 48.638, 60.983, 191.359])
patched_mean = np.array([45.980, 60.805, 48.793, 61.174, 190.511])

clean_std    = np.array([1.867, 0.451, 0.946, 1.483, 1.870])
patched_std  = np.array([1.696, 0.294, 0.402, 1.131, 2.060])

x = np.arange(len(models))
w = 0.36

# ------------------------------------------------------------
# Style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ------------------------------------------------------------
# SHORTER figure height
# ------------------------------------------------------------
fig, (ax_top, ax_bot) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(7.0, 3.5),   # shorter height
    gridspec_kw={"height_ratios": [0.9, 3.1], "hspace": 0.05}
)

# ------------------------------------------------------------
# Draw bars on both axes
# ------------------------------------------------------------
for ax in (ax_top, ax_bot):
    ax.bar(
        x - w/2, clean_mean,
        width=w,
        yerr=clean_std,
        capsize=3,
        label="Clean",
        color="#BFD7EA",
        edgecolor="#2E6F9E",
        linewidth=0.8
    )
    ax.bar(
        x + w/2, patched_mean,
        width=w,
        yerr=patched_std,
        capsize=3,
        label="Patched",
        color="#F6C7B6",
        edgecolor="#B64A2E",
        linewidth=0.8
    )
    ax.grid(axis="y", linestyle="--", alpha=0.3)

# ------------------------------------------------------------
# Broken y-axis limits
# ------------------------------------------------------------
ax_bot.set_ylim(0, 80)
ax_top.set_ylim(180, 202)

# lower axis: 10-by-10, but stop at 70 so the "80" near the break disappears
ax_bot.set_yticks(np.arange(0, 80, 10))   # 0,10,...,70
ax_top.set_yticks([180, 190, 200])

# ------------------------------------------------------------
# Hide connecting spines
# ------------------------------------------------------------
ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

# ------------------------------------------------------------
# Break marks
# ------------------------------------------------------------
d = 0.012
kwargs = dict(transform=ax_top.transAxes, color="k", clip_on=False, linewidth=1.1)
ax_top.plot((-d, +d), (-d, +d), **kwargs)
ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)

kwargs.update(transform=ax_bot.transAxes)
ax_bot.plot((-d, +d), (1 - d, 1 + d), **kwargs)
ax_bot.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

# ------------------------------------------------------------
# Labels / legend
# ------------------------------------------------------------
ax_bot.set_ylabel("End-to-end latency (ms)")
ax_bot.set_xticks(x)
ax_bot.set_xticklabels(models, rotation=20, ha="right")
ax_top.legend(frameon=False, loc="upper left")

# ------------------------------------------------------------
# Delta annotations
# ------------------------------------------------------------
for i in range(len(models)):
    delta = patched_mean[i] - clean_mean[i]
    y_text = max(clean_mean[i] + clean_std[i], patched_mean[i] + patched_std[i]) + 3

    if clean_mean[i] > 100:
        ax = ax_top
        y_plot = min(y_text, 199.2)
    else:
        ax = ax_bot
        y_plot = y_text

    ax.text(
        i, y_plot,
        f"Δ {delta:+.1f} ms",
        ha="center", va="bottom",
        fontsize=8.5, color="#333333"
    )

# ------------------------------------------------------------
# Final layout (no title, no bottom caption)
# ------------------------------------------------------------
plt.tight_layout()

plt.savefig(os.path.join(OUT_DIR, "hardware_e2e_clean_vs_patched_shorter.png"), bbox_inches="tight")
plt.savefig(os.path.join(OUT_DIR, "hardware_e2e_clean_vs_patched_shorter.pdf"), bbox_inches="tight")
plt.show()

print("Saved PNG:", os.path.join(OUT_DIR, "hardware_e2e_clean_vs_patched_shorter.png"))
print("Saved PDF:", os.path.join(OUT_DIR, "hardware_e2e_clean_vs_patched_shorter.pdf"))

In [ ]:
# ------------------------------------------------------------
# Broken ranges
# ------------------------------------------------------------
ax_bottom.set_ylim(0, 80)
ax_top.set_ylim(180, 200)

# y-axis ticks
ax_bottom.set_yticks(np.arange(0, 80, 10))   # 0 to 70 only
ax_top.set_yticks(np.arange(180, 201, 10))   # 180, 190, 200

# Hide spines between axes
ax_top.spines["bottom"].set_visible(False)
ax_bottom.spines["top"].set_visible(False)
ax_top.tick_params(labeltop=False, bottom=False)
ax_bottom.tick_params(top=False)

# Diagonal break marks
d = 0.008
kwargs = dict(transform=ax_top.transAxes, color="k", clip_on=False, linewidth=1.0)
ax_top.plot((-d, +d), (-d, +d), **kwargs)
ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)

kwargs.update(transform=ax_bottom.transAxes)
ax_bottom.plot((-d, +d), (1 - d, 1 + d), **kwargs)
ax_bottom.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

In [ ]:
# ============================================================
# Clean vs Patched E2E by model
# Broken y-axis, compact version, centered y-axis label
# Saves PNG + PDF
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
OUT_DIR = "/content/hardware_plots_cleaner"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ------------------------------------------------------------
# Data
# ------------------------------------------------------------
models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet+", "RT-DETR-L"]

clean_e2e = np.array([
    [47.442, 50.353, 45.077, 45.092, 43.782, 45.265, 45.453, 44.484, 44.953],
    [61.750, 60.870, 60.916, 60.319, 60.893, 60.860, 60.951, 60.019, 61.036],
    [47.872, 49.031, 49.958, 47.534, 47.843, 50.065, 48.315, 47.679, 49.447],
    [64.557, 60.628, 60.614, 60.284, 61.066, 61.408, 61.595, 58.918, 59.778],
    [193.543, 192.135, 190.627, 192.146, 188.355, 190.670, 189.642, 194.750, 190.362],
])

patched_e2e = np.array([
    [44.411, 50.044, 45.570, 47.676, 44.551, 45.768, 45.301, 44.893, 45.604],
    [60.771, 61.156, 60.774, 60.818, 60.682, 60.810, 61.042, 60.099, 61.097],
    [49.331, 48.310, 49.100, 48.744, 48.212, 49.284, 48.396, 49.086, 48.675],
    [63.412, 60.003, 61.718, 60.392, 61.001, 60.406, 60.143, 60.773, 62.719],
    [192.164, 190.977, 187.323, 194.243, 191.581, 187.776, 191.380, 189.717, 189.438],
])

clean_mean = clean_e2e.mean(axis=1)
patched_mean = patched_e2e.mean(axis=1)
clean_std = clean_e2e.std(axis=1)
patched_std = patched_e2e.std(axis=1)

# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------
x = np.arange(len(models))
w = 0.34

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(7.0, 3.6),
    gridspec_kw={"height_ratios": [1, 2], "hspace": 0.05}
)

fig.patch.set_facecolor("white")

clean_fill = "#BFD7EA"
clean_edge = "#2E6F9E"
patched_fill = "#F6C7B6"
patched_edge = "#B64A2E"

# ------------------------------------------------------------
# Draw bars on both axes
# ------------------------------------------------------------
for ax in (ax_top, ax_bot):
    ax.bar(
        x - w / 2,
        clean_mean,
        width=w,
        yerr=clean_std,
        capsize=2.5,
        label="Clean",
        color=clean_fill,
        edgecolor=clean_edge,
        linewidth=0.8,
        zorder=3
    )

    ax.bar(
        x + w / 2,
        patched_mean,
        width=w,
        yerr=patched_std,
        capsize=2.5,
        label="Patched",
        color=patched_fill,
        edgecolor=patched_edge,
        linewidth=0.8,
        zorder=3
    )

    ax.grid(axis="y", linestyle="--", alpha=0.25, zorder=0)

# ------------------------------------------------------------
# Broken y-axis limits
# ------------------------------------------------------------
ax_bot.set_ylim(35, 75)
ax_top.set_ylim(175, 205)

ax_bot.set_yticks([40, 50, 60, 70])
ax_top.set_yticks([180, 190, 200])

# Remove touching spines
ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)

ax_top.tick_params(labeltop=False, bottom=False)
ax_bot.tick_params(top=False)

# ------------------------------------------------------------
# Diagonal break marks
# ------------------------------------------------------------
d = 0.008

kwargs = dict(
    transform=ax_top.transAxes,
    color="black",
    clip_on=False,
    linewidth=1.0
)
ax_top.plot((-d, +d), (-d, +d), **kwargs)
ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)

kwargs.update(transform=ax_bot.transAxes)
ax_bot.plot((-d, +d), (1 - d, 1 + d), **kwargs)
ax_bot.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

# ------------------------------------------------------------
# Delta annotations
# ------------------------------------------------------------
for i in range(len(models)):
    delta = patched_mean[i] - clean_mean[i]
    y_text = max(clean_mean[i] + clean_std[i], patched_mean[i] + patched_std[i]) + 1.2

    if i < 4:
        ax_bot.text(
            i,
            y_text,
            f"Δ {delta:+.1f} ms",
            ha="center",
            va="bottom",
            fontsize=8.2,
            color="#333333"
        )
    else:
        ax_top.text(
            i,
            y_text,
            f"Δ {delta:+.1f} ms",
            ha="center",
            va="bottom",
            fontsize=8.2,
            color="#333333"
        )

# ------------------------------------------------------------
# Labels / legend
# ------------------------------------------------------------
ax_bot.set_xticks(x)
ax_bot.set_xticklabels(models, rotation=0, ha="center")

# Centered y-axis label across both broken axes
fig.text(
    0.025,
    0.50,
    "End-to-end latency (ms)",
    va="center",
    ha="center",
    rotation="vertical",
    fontsize=10
)

ax_top.legend(
    frameon=False,
    loc="upper left",
    ncol=2,
    bbox_to_anchor=(0.00, 1.04)
)

ax_bot.margins(x=0.06)

# ------------------------------------------------------------
# Layout and save BEFORE plt.show()
# ------------------------------------------------------------
png_path = os.path.join(
    OUT_DIR,
    "hardware_e2e_clean_vs_patched_by_model_compact.png"
)

pdf_path = os.path.join(
    OUT_DIR,
    "hardware_e2e_clean_vs_patched_by_model_compact.pdf"
)

fig.tight_layout(rect=[0.06, 0.00, 1.00, 1.00])

fig.savefig(png_path, format="png", bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, format="pdf", bbox_inches="tight", facecolor="white")

plt.show()

print("Saved PNG:", png_path)
print("Saved PDF:", pdf_path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
OUT_DIR = "/content/hardware_plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Data: Power visibility margin = |ΔP| / 0.22 W
# ------------------------------------------------------------
models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet+", "RT-DETR-L"]
scenarios = [f"B{i:02d}" for i in range(1, 10)]

power_margin = np.array([
    [0.15, 0.06, 0.11, 0.27, 0.30, 0.01, 0.15, 0.51, 0.00],
    [0.38, 0.28, 0.08, 0.41, 0.05, 0.02, 0.19, 0.04, 0.01],
    [0.00, 0.02, 0.00, 0.00, 0.00, 0.00, 0.02, 0.00, 0.12],
    [0.20, 0.00, 0.02, 0.06, 0.10, 0.05, 0.00, 0.22, 0.13],
    [0.28, 0.40, 1.89, 0.05, 0.43, 0.39, 0.35, 0.21, 0.11],
])

# ------------------------------------------------------------
# Style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 12,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

cmap = LinearSegmentedColormap.from_list(
    "power_margin_map",
    ["#0B7D3B", "#36A853", "#C7DB7A", "#F3C86B", "#E95C3B", "#C8102E"]
)

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.2, 3.2))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

im = ax.imshow(power_margin, cmap=cmap, vmin=0.0, vmax=2.0, aspect="auto")

# ticks and labels
ax.set_xticks(np.arange(len(scenarios)))
ax.set_xticklabels(scenarios)
ax.set_yticks(np.arange(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel("Billboard scenario")

# white cell borders
ax.set_xticks(np.arange(-0.5, len(scenarios), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(models), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
ax.tick_params(which="minor", bottom=False, left=False)

# ------------------------------------------------------------
# Annotate cells
# ------------------------------------------------------------
for i in range(power_margin.shape[0]):
    for j in range(power_margin.shape[1]):
        val = power_margin[i, j]

        if val > 1.0:
            ax.text(
                j, i, f"{val:.2f}",
                ha="center", va="center",
                fontsize=10.5, fontweight="bold", color="white"
            )
        else:
            ax.text(
                j, i, f"{val:.2f}",
                ha="center", va="center",
                fontsize=8.5, color="#1A1A1A", alpha=0.9
            )

# ------------------------------------------------------------
# Highlight only the outlier cell: RT-DETR-L / B03
# ------------------------------------------------------------
outlier_row, outlier_col = 4, 2

rect = Rectangle(
    (outlier_col - 0.5, outlier_row - 0.5),
    1, 1,
    fill=False,
    edgecolor="black",
    linewidth=2.5
)
ax.add_patch(rect)

# ------------------------------------------------------------
# Colorbar
# ------------------------------------------------------------
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Margin ratio")
cbar.outline.set_visible(False)

# threshold line at 1.0
cbar.ax.axhline(1.0, color="black", linestyle="--", linewidth=1.2)

# ------------------------------------------------------------
# Save BEFORE plt.show()
# ------------------------------------------------------------
png_path = os.path.join(OUT_DIR, "power_visibility_margin_heatmap_clean.png")
pdf_path = os.path.join(OUT_DIR, "power_visibility_margin_heatmap_clean.pdf")

fig.tight_layout()

fig.savefig(png_path, format="png", bbox_inches="tight", facecolor="white")
fig.savefig(pdf_path, format="pdf", bbox_inches="tight", facecolor="white")

plt.show()

print("Saved PNG:", png_path)
print("Saved PDF:", pdf_path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
OUT_DIR = "/content/hardware_plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Data: YOLOv8n pipeline breakdown
# ------------------------------------------------------------
labels = ["Clean", "Patched"]
preprocessing = [26.4, 28.3]
engine = [37.0, 37.0]
nms = [5.0, 5.3]

totals = np.array(preprocessing) + np.array(engine) + np.array(nms)
delta_total = totals[1] - totals[0]

# ------------------------------------------------------------
# Style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 16,
    "legend.fontsize": 14,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

C_PRE = "#C7D3DF"
C_ENG = "#6F96BD"
C_NMS = "#2B5A88"

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.2, 3.2))
fig.patch.set_facecolor("white")

ypos = np.array([1, 0])   # Clean on top, Patched below
bar_h = 0.38

# stacked horizontal bars
ax.barh(ypos, preprocessing, height=bar_h, color=C_PRE, edgecolor="white", linewidth=1.0)
ax.barh(ypos, engine, left=preprocessing, height=bar_h, color=C_ENG, edgecolor="white", linewidth=1.0)
ax.barh(ypos, nms, left=np.array(preprocessing) + np.array(engine), height=bar_h, color=C_NMS, edgecolor="white", linewidth=1.0)

# segment labels
for i in range(2):
    # preprocessing
    ax.text(preprocessing[i] / 2, ypos[i], f"{preprocessing[i]:.1f}",
            ha="center", va="center", color="#222222", fontsize=13)

    # engine
    ax.text(preprocessing[i] + engine[i] / 2, ypos[i], f"{engine[i]:.1f}",
            ha="center", va="center", color="white", fontsize=13, fontweight="bold")

    # nms
    ax.text(preprocessing[i] + engine[i] + nms[i] / 2, ypos[i], f"{nms[i]:.1f}",
            ha="center", va="center", color="white", fontsize=13, fontweight="bold")

    # total label at bar end
    ax.text(totals[i] + 1.0, ypos[i], f"{totals[i]:.1f} ms",
            ha="left", va="center", fontsize=16, fontweight="bold", color="#111111")

# bracket showing delta
x_bracket = max(totals) + 2.7
tick_len = 0.5

y_top = ypos[0] - bar_h / 2
y_bottom = ypos[1] + bar_h / 2

# bracket
ax.plot([x_bracket, x_bracket], [y_bottom, y_top], color="#777777", lw=1.5)
ax.plot([x_bracket - tick_len, x_bracket], [y_top, y_top], color="#777777", lw=1.5)
ax.plot([x_bracket - tick_len, x_bracket], [y_bottom, y_bottom], color="#777777", lw=1.5)

# delta text
ax.text(x_bracket + 0.6, (y_top + y_bottom) / 2, f"+{delta_total:.1f} ms",
        ha="left", va="center", fontsize=16, fontweight="bold", color="#333333")

# axes formatting
ax.set_yticks(ypos)
ax.set_yticklabels(labels)
ax.set_xlabel("Latency (ms)")
ax.set_xlim(0, 80)
ax.set_xticks(np.arange(0, 81, 10))
ax.grid(axis="x", linestyle="--", linewidth=0.8, alpha=0.35)

ax.spines["left"].set_color("#333333")
ax.spines["bottom"].set_color("#333333")
ax.spines["left"].set_linewidth(1.1)
ax.spines["bottom"].set_linewidth(1.1)

# legend above
legend_handles = [
    Patch(facecolor=C_PRE, edgecolor="none", label="Preprocessing"),
    Patch(facecolor=C_ENG, edgecolor="none", label="Engine"),
    Patch(facecolor=C_NMS, edgecolor="none", label="NMS"),
]
ax.legend(handles=legend_handles, ncol=3, frameon=False,
          loc="upper center", bbox_to_anchor=(0.5, 1.20))

# IMPORTANT: no title, no bottom italic sentence
plt.tight_layout()
plt.subplots_adjust(top=0.80)

plt.savefig(os.path.join(OUT_DIR, "yolov8n_pipeline_breakdown_clean.png"), bbox_inches="tight")
plt.savefig(os.path.join(OUT_DIR, "yolov8n_pipeline_breakdown_clean.pdf"), bbox_inches="tight")
plt.show()
# IMPORTANT: no title, no bottom italic sentence
plt.tight_layout()
plt.subplots_adjust(top=0.80)

png_path = os.path.join(OUT_DIR, "yolov8n_pipeline_breakdown_final.png")
pdf_path = os.path.join(OUT_DIR, "yolov8n_pipeline_breakdown_final.pdf")

plt.savefig(png_path, format="png", bbox_inches="tight", dpi=300)
plt.savefig(pdf_path, format="pdf", bbox_inches="tight")

plt.show()

print("Saved PNG:", png_path)
print("Saved PDF:", pdf_path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
OUT_DIR = "/content/hardware_plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Data: Power visibility margin = |ΔP| / 0.22 W
# ------------------------------------------------------------
models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet+", "RT-DETR-L"]
scenarios = [f"B{i:02d}" for i in range(1, 10)]

power_margin = np.array([
    [0.15, 0.06, 0.11, 0.27, 0.30, 0.01, 0.15, 0.51, 0.00],  # YOLOv8n
    [0.38, 0.28, 0.08, 0.41, 0.05, 0.02, 0.19, 0.04, 0.01],  # YOLOv8s
    [0.00, 0.02, 0.00, 0.00, 0.00, 0.00, 0.02, 0.00, 0.12],  # YOLOv5n
    [0.20, 0.00, 0.02, 0.06, 0.10, 0.05, 0.00, 0.22, 0.13],  # NanoDet+
    [0.28, 0.40, 1.89, 0.05, 0.43, 0.39, 0.35, 0.21, 0.11],  # RT-DETR-L
])

# ------------------------------------------------------------
# Style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 12,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# green -> yellow -> red
cmap = LinearSegmentedColormap.from_list(
    "power_margin_map",
    ["#0B7D3B", "#36A853", "#C7DB7A", "#F3C86B", "#E95C3B", "#C8102E"]
)

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.2, 3.2))
fig.patch.set_facecolor("white")

im = ax.imshow(power_margin, cmap=cmap, vmin=0.0, vmax=2.0, aspect="auto")

# ticks and labels
ax.set_xticks(np.arange(len(scenarios)))
ax.set_xticklabels(scenarios)
ax.set_yticks(np.arange(len(models)))
ax.set_yticklabels(models)
ax.set_xlabel("Billboard scenario")

# Optional title:
# ax.set_title(r"Power Visibility Margin: $|\Delta P| / 0.22$ W", pad=8)

# white cell borders
ax.set_xticks(np.arange(-0.5, len(scenarios), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(models), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
ax.tick_params(which="minor", bottom=False, left=False)

# ------------------------------------------------------------
# Annotate cells
# ------------------------------------------------------------
for i in range(power_margin.shape[0]):
    for j in range(power_margin.shape[1]):
        val = power_margin[i, j]

        if val > 1.0:
            ax.text(j, i, f"{val:.2f}",
                    ha="center", va="center",
                    fontsize=10.5, fontweight="bold", color="white")
        else:
            ax.text(j, i, f"{val:.2f}",
                    ha="center", va="center",
                    fontsize=8.5, color="#1A1A1A", alpha=0.9)

# ------------------------------------------------------------
# Highlight only the outlier cell (RT-DETR-L, B03)
# ------------------------------------------------------------
outlier_row, outlier_col = 4, 2
rect = Rectangle((outlier_col - 0.5, outlier_row - 0.5), 1, 1,
                 fill=False, edgecolor="black", linewidth=2.5)
ax.add_patch(rect)

# ------------------------------------------------------------
# Colorbar
# ------------------------------------------------------------
cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Margin ratio")
cbar.outline.set_visible(False)

# threshold line at 1.0
threshold = 1.0
cbar.ax.axhline(threshold, color="black", linestyle="--", linewidth=1.2)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "power_visibility_margin_heatmap_clean.png"), bbox_inches="tight")
plt.savefig(os.path.join(OUT_DIR, "power_visibility_margin_heatmap_clean.pdf"), bbox_inches="tight")
plt.show()

print("Saved:", os.path.join(OUT_DIR, "power_visibility_margin_heatmap_clean.png"))


plt.tight_layout()

plt.savefig(os.path.join(OUT_DIR, "power_visibility_margin_heatmap_final.pdf"),
            format="pdf", bbox_inches="tight")
plt.savefig(os.path.join(OUT_DIR, "power_visibility_margin_heatmap_clean.png"),
            format="png", bbox_inches="tight")

plt.show()

print("Saved PDF:", os.path.join(OUT_DIR, "power_visibility_margin_heatmap_clean.pdf"))
print("Saved PNG:", os.path.join(OUT_DIR, "power_visibility_margin_heatmap_clean.png"))

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
OUT_DIR = "/content/hardware_plots"
os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Data: Latency visibility margin = |ΔE2E| / sigma_clean
# ------------------------------------------------------------
models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "NanoDet+", "RT-DETR-L"]
scenarios = [f"B{i:02d}" for i in range(1, 10)]

latency_margin = np.array([
    [0.49, 0.06, 0.14, 0.72, 0.29, 0.13, 0.05, 0.11, 0.20],
    [0.24, 0.44, 0.19, 0.43, 0.29, 0.08, 0.14, 0.06, 0.09],
    [0.58, 0.21, 0.24, 0.44, 0.12, 0.21, 0.03, 0.49, 0.22],
    [0.13, 0.13, 0.24, 0.02, 0.01, 0.20, 0.28, 0.62, 0.69],
    [0.10, 0.09, 0.23, 0.14, 0.23, 0.21, 0.12, 0.37, 0.06],
])

# ------------------------------------------------------------
# Style
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 12,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

cmap = LinearSegmentedColormap.from_list(
    "latency_margin_map",
    ["#0B7D3B", "#36A853", "#C7DB7A", "#F3C86B", "#E95C3B", "#C8102E"]
)

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9.2, 3.2))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

im = ax.imshow(latency_margin, cmap=cmap, vmin=0.0, vmax=1.4, aspect="auto")

# ticks and labels
ax.set_xticks(np.arange(len(scenarios)))
ax.set_xticklabels(scenarios)

ax.set_yticks(np.arange(len(models)))
ax.set_yticklabels(models)

ax.set_xlabel("Billboard scenario")

# white cell borders
ax.set_xticks(np.arange(-0.5, len(scenarios), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(models), 1), minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
ax.tick_params(which="minor", bottom=False, left=False)

# ------------------------------------------------------------
# Annotate cells
# ------------------------------------------------------------
for i in range(latency_margin.shape[0]):
    for j in range(latency_margin.shape[1]):
        val = latency_margin[i, j]
        ax.text(
            j, i, f"{val:.2f}",
            ha="center", va="center",
            fontsize=8.5,
            color="#1A1A1A",
            alpha=0.9
        )

# ------------------------------------------------------------
# Colorbar
# ------------------------------------------------------------
cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Margin ratio")
cbar.outline.set_visible(False)

# threshold line at 1.0
cbar.ax.axhline(1.0, color="black", linestyle="--", linewidth=1.2)

# ------------------------------------------------------------
# Save ONCE before plt.show()
# ------------------------------------------------------------
png_path = os.path.join(OUT_DIR, "latency_visibility_margin_heatmap_clean.png")
pdf_path = os.path.join(OUT_DIR, "latency_visibility_margin_heatmap_clean.pdf")

fig.tight_layout()

fig.savefig(png_path, format="png", bbox_inches="tight", dpi=300, facecolor="white")
fig.savefig(pdf_path, format="pdf", bbox_inches="tight", facecolor="white")

plt.show()

print("Saved PNG:", png_path)
print("Saved PDF:", pdf_path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 180
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.family"] = "DejaVu Serif"
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10

OUTDIR = "thesis_figures"
os.makedirs(OUTDIR, exist_ok=True)

def savefig(fig, name):
    png = os.path.join(OUTDIR, f"{name}.png")
    pdf = os.path.join(OUTDIR, f"{name}.pdf")
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ---------- style ----------
plt.rcParams["figure.dpi"] = 180
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.family"] = "DejaVu Serif"
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10
plt.rcParams["xtick.labelsize"] = 9
plt.rcParams["ytick.labelsize"] = 10

OUTDIR = "thesis_figures"
os.makedirs(OUTDIR, exist_ok=True)

def savefig(fig, name):
    png = os.path.join(OUTDIR, f"{name}.png")
    pdf = os.path.join(OUTDIR, f"{name}.pdf")
    fig.savefig(png, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf, bbox_inches="tight", facecolor="white")
    print("Saved:", png)
    print("Saved:", pdf)

# ---------- data ----------
scenes = ["B01","B02","B03","B04","B05","B06","B07","B08","B09"]

delta_map50 = np.array([-0.094, -0.364, -0.404, -0.659, -0.145, -0.186, -0.124, -0.454, -0.402])
delta_recall = np.array([-0.199, -0.245, -0.400, -0.572, -0.256, -0.313, -0.109, -0.453, -0.250])
delta_fp001 = np.array([48.52, 50.26, 53.04, 49.28, 33.16, 34.42, 39.48, 52.16, 35.68])

# convert drops to positive magnitudes for cleaner plotting
map_drop = -delta_map50
recall_drop = -delta_recall
fp_increase = delta_fp001

y = np.arange(len(scenes))

# ---------- figure ----------
fig, axes = plt.subplots(1, 3, figsize=(11.2, 4.5), sharey=True)
fig.patch.set_facecolor("white")

bar_height = 0.56

panels = [
    (axes[0], map_drop, "mAP50 drop", r"$-\Delta$ mAP50", 0.72, "{:.2f}"),
    (axes[1], recall_drop, "Recall drop", r"$-\Delta$ Recall", 0.62, "{:.2f}"),
    (axes[2], fp_increase, "FP increase", r"$\Delta$ FP/img@0.001", 58, "{:.1f}")
]

for ax, values, title, xlabel, xmax, fmt in panels:
    ax.set_facecolor("white")
    ax.barh(y, values, height=bar_height)
    ax.set_title(title, pad=8)
    ax.set_xlabel(xlabel)
    ax.set_xlim(0, xmax)
    ax.grid(axis="x", linewidth=0.6, alpha=0.25)
    ax.set_axisbelow(True)

    # clean spines
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_visible(False)

    # value labels
    offset = xmax * 0.012
    for i, v in enumerate(values):
        ax.text(v + offset, i, fmt.format(v), va="center", ha="left", fontsize=9)

axes[0].set_yticks(y)
axes[0].set_yticklabels(scenes)
axes[0].invert_yaxis()

fig.suptitle("YOLOv8n per-scenario attack severity", y=0.98, fontsize=13)
plt.tight_layout(rect=[0, 0, 1, 0.95])

savefig(fig, "fig_yolov8n_per_scenario_clean")
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ---------- output ----------
OUTDIR = "thesis_figures"
os.makedirs(OUTDIR, exist_ok=True)

def savefig(fig, name):
    png = os.path.join(OUTDIR, f"{name}.png")
    pdf = os.path.join(OUTDIR, f"{name}.pdf")
    fig.savefig(png, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf, bbox_inches="tight", facecolor="white")
    print("Saved:", png)
    print("Saved:", pdf)

# ---------- thesis style ----------
plt.rcParams.update({
    "figure.dpi": 180,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "DejaVu Serif",
    "font.size": 9,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9
})

# ---------- thesis palette ----------
COLORS = {
    "clean": "#4C72B0",       # blue
    "patched": "#C44E52",     # red
    "transferred": "#55A868", # green
    "random": "#8C8C8C"       # gray
}

# ---------- data ----------
models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "RT-DETR-L", "NanoDet+"]
clean_map50 = np.array([80.6, 89.0, 86.1, 90.4, 64.6])
patched_map50 = np.array([49.2, 63.2, 56.7, 67.8, 58.4])
delta_map50 = patched_map50 - clean_map50  # negative values

x = np.arange(len(models))
w = 0.32

# ---------- plot ----------
fig, ax = plt.subplots(figsize=(8.0, 4.6))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

b1 = ax.bar(x - w/2, clean_map50, w, label="Clean", color=COLORS["clean"])
b2 = ax.bar(x + w/2, patched_map50, w, label="Patched", color=COLORS["patched"])

# labels on bars
for bars in (b1, b2):
    ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=8)

# cleaner delta labels
for i, d in enumerate(delta_map50):
    top = max(clean_map50[i], patched_map50[i])
    ax.text(x[i], top + 8.0, f"Δ {d:.1f}", ha="center", va="bottom", fontsize=9)

# axes / grid / styling
ax.set_title("Cross-model transferability", pad=10)
ax.set_ylabel("mAP50 (%)")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 102)
ax.grid(axis="y", linestyle="--", alpha=0.30)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False, loc="upper right")

plt.tight_layout()
savefig(fig, "fig_cross_model_transfer_map50")
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ---------- output ----------
OUTDIR = "thesis_figures"
os.makedirs(OUTDIR, exist_ok=True)

def savefig(fig, name):
    png = os.path.join(OUTDIR, f"{name}.png")
    pdf = os.path.join(OUTDIR, f"{name}.pdf")
    fig.savefig(png, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf, bbox_inches="tight", facecolor="white")
    print("Saved:", png)
    print("Saved:", pdf)

# ---------- thesis style ----------
plt.rcParams.update({
    "figure.dpi": 180,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "DejaVu Serif",
    "font.size": 9,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9
})

# ---------- thesis palette ----------
COLORS = {
    "clean": "#4C72B0",
    "patched": "#C44E52",
}

# ---------- data ----------
models = ["YOLOv8n", "YOLOv8s", "YOLOv5n", "RT-DETR-L", "NanoDet+"]
clean_fp03 = np.array([0.5467, 0.5044, 0.5556, 0.8556, 1.2889])
patched_fp03 = np.array([20.0822, 0.8778, 2.4178, 2.7911, 1.8200])
delta_fp03 = patched_fp03 - clean_fp03

x = np.arange(len(models))
w = 0.32

# ---------- figure ----------
fig, (ax_top, ax_bot) = plt.subplots(
    2, 1,
    sharex=True,
    figsize=(8.1, 5.0),
    gridspec_kw={"height_ratios": [1, 3], "hspace": 0.04}
)

fig.patch.set_facecolor("white")
for ax in (ax_top, ax_bot):
    ax.set_facecolor("white")
    ax.bar(x - w/2, clean_fp03, w, color=COLORS["clean"], label="Clean")
    ax.bar(x + w/2, patched_fp03, w, color=COLORS["patched"], label="Patched")
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# ---------- improved break ranges ----------
ax_bot.set_ylim(0, 3.4)
ax_top.set_ylim(18.0, 21.8)

ax_bot.set_yticks(np.arange(0, 3.5, 0.5))
ax_top.set_yticks([18, 19, 20, 21])

# hide touching spines
ax_top.spines["bottom"].set_visible(False)
ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(axis="x", which="both", bottom=False, labelbottom=False)

# ---------- cleaner break marks ----------
d = 0.008
kwargs = dict(color="k", clip_on=False, linewidth=0.9)

# top axis break marks
ax_top.plot((-d, +d), (-d, +d), transform=ax_top.transAxes, **kwargs)
ax_top.plot((1 - d, 1 + d), (-d, +d), transform=ax_top.transAxes, **kwargs)

# bottom axis break marks
ax_bot.plot((-d, +d), (1 - d, 1 + d), transform=ax_bot.transAxes, **kwargs)
ax_bot.plot((1 - d, 1 + d), (1 - d, 1 + d), transform=ax_bot.transAxes, **kwargs)

# ---------- labels ----------
ax_top.set_title("Cross-model false positives at conf ≥ 0.3", pad=10)
ax_bot.set_ylabel("FP/img @ 0.3")
ax_bot.set_xticks(x)
ax_bot.set_xticklabels(models)

# legend
ax_top.legend(frameon=False, loc="upper right")

# value labels
for i in range(len(models)):
    # clean labels always on bottom axis
    ax_bot.text(x[i] - w/2, clean_fp03[i] + 0.07, f"{clean_fp03[i]:.2f}",
                ha="center", va="bottom", fontsize=8)

    # patched labels
    if patched_fp03[i] > ax_bot.get_ylim()[1]:
        ax_top.text(x[i] + w/2, patched_fp03[i] + 0.06, f"{patched_fp03[i]:.2f}",
                    ha="center", va="bottom", fontsize=8)
        ax_top.text(x[i], patched_fp03[i] + 0.55, f"Δ +{delta_fp03[i]:.2f}",
                    ha="center", va="bottom", fontsize=9)
    else:
        ax_bot.text(x[i] + w/2, patched_fp03[i] + 0.07, f"{patched_fp03[i]:.2f}",
                    ha="center", va="bottom", fontsize=8)
        ax_bot.text(x[i], patched_fp03[i] + 0.34, f"Δ +{delta_fp03[i]:.2f}",
                    ha="center", va="bottom", fontsize=9)

plt.tight_layout()
savefig(fig, "fig_cross_model_fp03_better_break")
plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Output folder
# ============================================================
OUTDIR = "thesis_figures"
os.makedirs(OUTDIR, exist_ok=True)

def savefig(fig, name):
    png = os.path.join(OUTDIR, f"{name}.png")
    pdf = os.path.join(OUTDIR, f"{name}.pdf")
    fig.savefig(png, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf, bbox_inches="tight", facecolor="white")
    print("Saved:", png)
    print("Saved:", pdf)

# ============================================================
# Style
# ============================================================
plt.rcParams.update({
    "figure.dpi": 180,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "font.family": "DejaVu Serif",
    "font.size": 9,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9
})

# ============================================================
# Thesis palette
# ============================================================
COLORS = {
    "Clean": "#4C72B0",       # blue
    "Random": "#9A9A9A",      # gray
    "Optimized": "#C44E52"    # red
}

conditions = ["Clean", "Random", "Optimized"]
x = np.arange(len(conditions))
bar_colors = [COLORS[c] for c in conditions]

In [ ]:
# ============================================================
# FP/img @ 0.5  -- match the FP/img @ 0.001 style
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'font.size': 10,
    'axes.titlesize': 15,
    'axes.labelsize': 12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11
})

conditions = ["Clean", "Random", "Optimized"]
vals = np.array([0.26, 0.13, 3.08])
errs = np.array([0.18, 0.12, 4.15])

x = np.arange(len(conditions))
bar_colors = ['#4C72B0', '#9E9E9E', '#C44E52']

fig, ax = plt.subplots(figsize=(5.6, 4.4))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

bars = ax.bar(
    x, vals,
    width=0.62,
    color=bar_colors,
    edgecolor='none',
    yerr=errs,
    capsize=5,
    error_kw=dict(elinewidth=1.6, capthick=1.3)
)

# value labels on bars
for b, v in zip(bars, vals):
    ax.text(
        b.get_x() + b.get_width()/2,
        b.get_height() + 0.08,
        f"{v:.2f}",
        ha='center', va='bottom', fontsize=12
    )

# axes
ax.set_title("False positives per image at conf ≥ 0.5", pad=8)
ax.set_ylabel("FP/img @ 0.5")
ax.set_xticks(x)
ax.set_xticklabels(conditions)
ax.set_ylim(0, 8)

# grid/spines
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# top-right delta text, same style as the 0.001 plot
delta = vals[2] - vals[0]
ax.text(
    0.97, 0.98,
    f"Δ vs Clean = +{delta:.2f}",
    transform=ax.transAxes,
    ha='right', va='top', fontsize=12
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# FP/img @ 0.5  -- cleaner text placement
# ============================================================
vals = np.array([0.26, 0.13, 3.08])
errs = np.array([0.18, 0.12, 4.15])

fig, ax = plt.subplots(figsize=(4.8, 3.6))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

bars = ax.bar(
    x, vals, width=0.62,
    color=bar_colors,
    edgecolor="none",
    yerr=errs,
    capsize=4
)

# bar value labels
for b, v in zip(bars, vals):
    ax.text(
        b.get_x() + b.get_width()/2,
        b.get_height() + 0.08,
        f"{v:.2f}",
        ha="center", va="bottom", fontsize=9
    )

ax.set_title("False positives per image at conf ≥ 0.5", pad=8)
ax.set_ylabel("FP/img @ 0.5")
ax.set_xticks(x)
ax.set_xticklabels(conditions)
ax.set_ylim(0, 8)

ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# cleaner summary box instead of floating middle text
delta = vals[2] - vals[0]
ratio = vals[2] / vals[0]

ax.text(
    0.97, 0.94,
    f"Optimized vs Clean\nΔ = +{delta:.2f} FP/img\n{ratio:.1f}× increase",
    transform=ax.transAxes,
    ha="right", va="top",
    fontsize=8.5,
    bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="0.8", alpha=0.95)
)

plt.tight_layout()
savefig(fig, "fp_img_conf_05_clean_text")
plt.show()

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
out = '/content/drive/MyDrive/plots'
os.makedirs(out, exist_ok=True)

plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'font.size': 7,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7
})

s = ['B02','B03','B04','B05','B06','B07','B08','B09']
x = np.arange(len(s))
w = 0.24
colors = ['#4C72B0', '#C44E52', '#55A868']

plots = {
    'transfer_fp_conf03.pdf': (
        [0.62,0.24,0.38,0.36,0.52,1.24,0.12,0.96],
        [17.32,18.64,19.34,24.16,22.64,17.92,15.24,21.96],
        [11.86,14.72,14.68,18.52,18.12,15.72,13.94,19.72],
        r'FP/img @ conf $\geq$ 0.3', 29
    ),
    'transfer_fp_conf05.pdf': (
        [0.42,0.10,0.14,0.18,0.18,0.66,0.06,0.28],
        [8.24,10.38,8.50,14.40,11.72,8.50,5.84,14.30],
        [6.12,9.12,8.76,12.80,11.98,11.00,7.18,12.32],
        r'FP/img @ conf $\geq$ 0.5', 16
    )
}

for name, (clean, own, trans, ylabel, ymax) in plots.items():
    fig, ax = plt.subplots(figsize=(7.2, 2.6))

    b1 = ax.bar(x-w, clean, w, label='Clean', color=colors[0])
    b2 = ax.bar(x,   own,   w, label='Scenario-specific', color=colors[1])
    b3 = ax.bar(x+w, trans, w, label='Transferred from B01', color=colors[2])

    for b in (b1, b2, b3):
        ax.bar_label(b, fmt='%.1f', padding=1, fontsize=6.2)

    ax.set(xlabel='Scenario', ylabel=ylabel, xticks=x, xticklabels=s, ylim=(0, ymax))
    ax.set_yticks(np.arange(0, ymax + 1, 5))
    ax.grid(axis='y', ls='--', alpha=0.35)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)

    plt.tight_layout()
    plt.savefig(f'{out}/{name}', bbox_inches='tight')
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("figures", exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"figures/{name}.pdf", bbox_inches="tight")
    plt.savefig(f"figures/{name}.png", bbox_inches="tight")
    print(f"Saved: figures/{name}.pdf and .png")

In [ ]:
# Data
defenses = ["None", "JPEG", "Bit-depth", "Gaussian blur", "Median filter", "LGS", "SODA"]
patched_map = np.array([0.254, 0.251, 0.254, 0.135, 0.163, 0.235, 0.231])
patched_fp  = np.array([63.780, 63.147, 63.847, 40.220, 34.553, 23.240, 31.090])

plt.figure(figsize=(7.2, 5.2))

# Plot all points
for i, name in enumerate(defenses):
    plt.scatter(patched_map[i], patched_fp[i], s=110, zorder=3)

# Individual labels for non-overlapping points
label_cfg = {
    "JPEG":          {"xytext": (8, -2),  "ha": "left",  "va": "top"},
    "Gaussian blur": {"xytext": (6, 6),   "ha": "left",  "va": "center"},
    "Median filter": {"xytext": (6, 6),   "ha": "left",  "va": "center"},
    "LGS":           {"xytext": (6, 2),   "ha": "left",  "va": "bottom"},
    "SODA":          {"xytext": (6, 4),   "ha": "left",  "va": "bottom"},
}

for i, name in enumerate(defenses):
    if name in label_cfg:
        cfg = label_cfg[name]
        plt.annotate(
            name,
            (patched_map[i], patched_fp[i]),
            xytext=cfg["xytext"],
            textcoords="offset points",
            ha=cfg["ha"],
            va=cfg["va"]
        )

# Shared label for None + Bit-depth
shared_x = 0.254
shared_y = (63.780 + 63.847) / 2

plt.annotate(
    "None / Bit-depth",
    (shared_x, shared_y),
    xytext=(8, 10),
    textcoords="offset points",
    ha="left",
    va="bottom"
)

plt.xlabel("Patched mAP50-95")
plt.ylabel("Patched FP/img")
plt.title("Held-out defense trade-off on billboard04--09")
plt.grid(True, alpha=0.25, zorder=0)

plt.xlim(0.129, 0.260)
plt.ylim(21, 66)

savefig("defense_tradeoff_scatter_clean")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("figures", exist_ok=True)

def savefig(name):
    plt.tight_layout()
    plt.savefig(f"figures/{name}.pdf", bbox_inches="tight")
    plt.savefig(f"figures/{name}.png", bbox_inches="tight")
    print(f"Saved: figures/{name}.pdf and .png")

# Correct raw FP/img values from your LaTeX table
defenses = np.array([
    "Patched", "JPEG", "Median", "LGS", "Gaussian", "SODA"
])
fpimg = np.array([
    386.42, 386.58, 370.44, 369.49, 365.92, 335.49
])

order = np.argsort(fpimg)
defenses_sorted = defenses[order]
fpimg_sorted = fpimg[order]

colors = []
for d in defenses_sorted:
    if d == "LGS":
        colors.append("tab:blue")
    elif d == "SODA":
        colors.append("tab:pink")
    elif d == "Patched":
        colors.append("tab:red")
    else:
        colors.append("0.55")

y = np.arange(len(defenses_sorted))

plt.figure(figsize=(8.6, 4.9))

for yi, val in zip(y, fpimg_sorted):
    plt.plot([0, val], [yi, yi], color="0.84", linewidth=2.2,
             solid_capstyle="round", zorder=1)

plt.scatter(fpimg_sorted, y, s=125, color=colors, zorder=3)

for yi, val in zip(y, fpimg_sorted):
    plt.text(val + 3, yi, f"{val:.2f}", va="center", ha="left", fontsize=10)

baseline = 386.42
plt.axvline(baseline, color="0.35", linestyle="--", linewidth=1.2)
plt.text(baseline + 2.5, len(defenses_sorted) - 0.35, "Patched baseline",
         fontsize=10, color="0.35")

plt.yticks(y, defenses_sorted)
plt.xlabel("FP/img on patched Isaac Sim test set")
plt.title("Isaac Sim defense comparison: raw false positives")
plt.grid(True, axis="x", alpha=0.18)

ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.xlim(320, 392)
savefig("isaac_defense_raw_fp_lollipop_corrected")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

thresholds = [0.3, 0.5, 0.7]

# Correct values from your LaTeX tables
fp_thr = {
    "Patched":  [23.73, 13.69, 5.56],
    "JPEG":     [24.08, 13.85, 5.61],   # keep only if these are confirmed from your evals
    "Median":   [22.03, 13.27, 5.93],   # keep only if confirmed
    "Gaussian": [21.39, 13.39, 6.32],   # keep only if confirmed
    "LGS":   [19.53, 11.02, 4.41],   # keep only if confirmed
    "SODA":     [10.51, 5.98, 2.44],
}

plt.figure(figsize=(7.6, 4.9))

for name, vals in fp_thr.items():
    lw = 2.6 if name == "SODA" else 2.0
    z = 4 if name == "SODA" else 2
    plt.plot(thresholds, vals, marker="o", linewidth=lw, label=name, zorder=z)

plt.xticks(thresholds, ["0.3", "0.5", "0.7"])
plt.xlabel("Confidence threshold")
plt.ylabel("FP/img")
plt.title("Isaac Sim operational false positives across thresholds")
plt.grid(True, alpha=0.25)
plt.legend()

savefig("isaac_operational_fp_thresholds_corrected")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

threshold_labels = ["0.3", "0.5", "0.7"]
patched = np.array([23.73, 13.69, 5.56])
soda    = np.array([10.51,  5.98, 2.44])

x = np.arange(len(threshold_labels))
w = 0.34

plt.figure(figsize=(7.2, 4.8))
bars1 = plt.bar(x - w/2, patched, width=w, label="Patched baseline")
bars2 = plt.bar(x + w/2, soda,    width=w, label="SODA")

plt.xticks(x, threshold_labels)
plt.xlabel("Confidence threshold")
plt.ylabel("FP/img")
plt.title("SODA suppression of operational false positives on Isaac Sim")
plt.grid(True, axis="y", alpha=0.25)
plt.legend()

for bars in [bars1, bars2]:
    for b in bars:
        h = b.get_height()
        plt.text(b.get_x() + b.get_width()/2, h + 0.35, f"{h:.2f}",
                 ha="center", va="bottom", fontsize=10)

plt.ylim(0, 27)
savefig("isaac_soda_operational_fp_bar")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

conditions = ["Patched", "Gaussian", "Median", "JPEG", "LGS", "SODA"]
det_img = np.array([460.36, 439.41, 443.12, 460.46, 443.53, 402.56])
fp_img  = np.array([386.42, 365.92, 370.44, 386.58, 369.49, 335.49])

x = np.arange(len(conditions))
w = 0.36

plt.figure(figsize=(9.2, 5.0))
bars1 = plt.bar(x - w/2, det_img, width=w, label="Det/img")
bars2 = plt.bar(x + w/2, fp_img,  width=w, label="FP/img")

plt.xticks(x, conditions, rotation=20)
plt.ylabel("Count per image")
plt.title("Isaac Sim defense effect on detector overload")
plt.grid(True, axis="y", alpha=0.25)
plt.legend()

for bars in [bars1, bars2]:
    for b in bars:
        h = b.get_height()
        plt.text(b.get_x() + b.get_width()/2, h + 3, f"{h:.2f}",
                 ha="center", va="bottom", fontsize=9)

plt.ylim(300, 480)
savefig("isaac_soda_overload_det_fp")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

conditions = ["Clean", "SODA"]
precision = [0.862, 0.880]
recall    = [0.740, 0.615]
map50     = [0.794, 0.681]
map5095   = [0.636, 0.468]

metrics = ["Precision", "Recall", "mAP50", "mAP50-95"]
vals = np.array([
    [precision[0], recall[0], map50[0], map5095[0]],
    [precision[1], recall[1], map50[1], map5095[1]],
])

x = np.arange(len(metrics))
w = 0.34

plt.figure(figsize=(8.4, 5.0))
plt.bar(x - w/2, vals[0], width=w, label="Clean")
plt.bar(x + w/2, vals[1], width=w, label="SODA")

plt.xticks(x, metrics)
plt.ylabel("Score")
plt.title("Detection quality after SODA defense on Isaac Sim")
plt.ylim(0, 1.0)
plt.grid(True, axis="y", alpha=0.25)
plt.legend()

savefig("isaac_soda_detection_quality")
plt.show()